# 02 — Estrategia de particionado

Cuándo particionar, cuándo no, y cuánto. Las reglas que aprendí a la
fuerza.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("02_02_particionado")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

df = spark.read.parquet("/home/jovyan/work/data/sample/ventas/parquet_limpio")

## Las 4 reglas de oro

1. **Baja cardinalidad** — idealmente < 1000 valores distintos por
   columna de partición.
2. **Distribución pareja** — si una partición tiene el 99% de los
   datos, perdiste el partition pruning.
3. **Filtraste** por esa columna casi siempre — si casi nunca filtrás,
   no la particiones.
4. **No sobre-particiones** — cada partición es un directorio con su
   metadata. 10K particiones chicas = dolor.

In [ ]:
print("Particiones actuales:")
df.select("anio", "mes").distinct().orderBy("anio", "mes").show(20)

print("Filas por partición (top 5):")
df.groupBy("anio", "mes").count().orderBy(F.col("count").desc()).show(5)

## Ejemplo: una buena partición vs una mala

**Buena**: `anio` y `mes`. Cardinalidad ~12 valores por año. Filtrás
constantemente por fecha.

**Mala**: `id` (cardinalidad = filas). Cada fila es una partición.
Archivos de 0 bytes. Catástrofe.

In [ ]:
# Demostración del costo de mal particionar
from pathlib import Path

BAD = "/home/jovyan/work/data/sample/ventas/bad_partition"
Path(BAD).mkdir(parents=True, exist_ok=True)

# No lo corras con 1M filas, explotaría el filesystem.
# Acá demostramos con un subconjunto de 10K filas.
df.limit(10_000).write.mode("overwrite").partitionBy("id").parquet(BAD)

n_dirs = sum(1 for _ in Path(BAD).rglob("*.parquet"))
print(f"Particiones creadas: {n_dirs}")
print("↑↑↑ NO HAGAS ESTO EN PRODUCCIÓN ↑↑↑")

## Repartition vs Coalesce

Para controlar el **número de archivos** sin cambiar la estructura
de particiones:

- `coalesce(N)`: reduce particiones **sin shuffle** (solo mergea
  archivos). Para outputs chicos, escribí con `coalesce(1)` o
  `coalesce(N)` donde N es chico.
- `repartition(N)`: hace shuffle. Para mejorar paralelismo en
  lecturas futuras.

In [ ]:
# Ejemplo: 200 particiones → 10 (sin shuffle)
df_small = df.coalesce(10)
print("Particiones después de coalesce(10):", df_small.rdd.getNumPartitions())

## ✅ Checkpoint

- [x] Entiendo cuándo particionar (baja cardinalidad, filtro
      frecuente, distribución pareja).
- [x] Sé la diferencia entre coalesce (sin shuffle) y repartition
      (con shuffle).
- [x] Reconozco cuándo NO particionar (columnas de alta cardinalidad).

**Siguiente**: `03_delta_lake_basico.ipynb` — Parquet + transacciones
ACID + time travel.